# Google ADK Foundations: From a Local Agent to AgentSpace

## 📚 Learning Objectives

In this notebook, you will learn how to:
- **Explain what Google ADK is** and how it relates to **Google AgentSpace**
- **Build a minimal ADK agent** (`LlmAgent`, aliased as `Agent`) with an instruction and a tool
- **Run the agent locally** using ADK's `Runner` + `InMemorySessionService` mechanism
- **Understand, conceptually, how an ADK agent gets packaged and deployed onto AgentSpace** for enterprise discovery and hosting

## 🎯 Where This Fits

This notebook lives in `06_Agent_SDKs_First_Party/Google_ADK/01_Foundations/` — the first-party, framework-native track for Google's own agent-building SDK, distinct from LangGraph/LangChain (Phases 2–4) and from CrewAI/AutoGen/DSPy (Phase 10). It follows the same narrative style as those framework tracks: it uses **ADK's own native agent/model configuration**, not this repo's `helpers.get_llm()` factory (that factory is reserved for LangGraph-phase notebooks).

## 🔑 Key Concepts

- **Google ADK (Agent Development Kit)**: Google's open-source Python (and Java) framework for building, testing, evaluating, and deploying AI agents — conceptually similar in role to LangGraph or CrewAI, but built by Google and designed to integrate tightly with Gemini models and Vertex AI.
- **Agent / `LlmAgent`**: An ADK agent driven by an LLM, configured with a `model`, an `instruction` (system prompt), and optional `tools`.
- **Tool**: A plain Python function ADK automatically wraps into a callable tool, based on its signature, type hints, and docstring.
- **Runner + `SessionService`**: The local execution harness that manages conversation state (`Session`) and drives the agent loop, emitting a stream of `Event` objects.
- **AgentSpace**: Google's enterprise agent-hosting and discovery platform, built on top of Vertex AI Agent Builder, where ADK agents (among others) get registered, secured, and surfaced to end users inside an organization.

## 🚀 Let's Get Started!

## 1. What is Google ADK, and how does it relate to AgentSpace?

### What we are going to do

Before writing any code, it's worth being precise about two names that are easy to conflate: **Google ADK** and **Google AgentSpace**. They sit at different layers of the same stack.

- **Google ADK (Agent Development Kit)** is an **open-source Python/Java framework** for *building* agents. You write code: agents, tools, sub-agents, callbacks, evaluation configs. It runs anywhere — your laptop, a container, Cloud Run, or Vertex AI — and it is model-agnostic in principle, though it is designed to work especially well with Gemini.
- **Google AgentSpace** is a **managed, enterprise-facing product** for *hosting, securing, and discovering* agents inside an organization. It sits on top of **Vertex AI Agent Builder** infrastructure. Instead of every employee needing to know which repo or endpoint hosts which agent, AgentSpace gives an organization a single, governed "storefront" where agents (built with ADK, or otherwise) are registered, permissioned per user/group, and made searchable/chatable through one enterprise UI.

**The relationship in one sentence:** ADK is the framework you use to *build* an agent's logic; AgentSpace is where a *finished* agent gets *deployed, governed, and exposed* to end users at enterprise scale — you do not need AgentSpace to build or run an ADK agent (as this notebook demonstrates), but AgentSpace is one of the primary places a production ADK agent ends up living.

This notebook covers the ADK side end-to-end (steps 2–3), and then covers the AgentSpace side conceptually (step 4), since a real AgentSpace deployment requires a provisioned GCP enterprise project, IAM/OAuth setup, and Agent Builder access that are out of scope for a local notebook.

## 2. Installing ADK and Building a Minimal Agent

### What we are going to do

We'll install the `google-adk` package, then build the smallest useful agent: an `Agent` (ADK's public alias for `LlmAgent`) configured with:
- a `model` (a Gemini model id string, resolved via the Gemini API when `GOOGLE_API_KEY` is set),
- an `instruction` (the system prompt that defines its behavior),
- and one plain Python function as a `tool`, which ADK automatically wraps into a callable tool based on its signature, type hints, and docstring — no manual schema authoring required.

This mirrors how the CrewAI/AutoGen notebooks in Phase 10 configure their own native `Agent`/`LLM` objects directly, rather than routing through this repo's `helpers.get_llm()` factory (that factory is specific to the LangGraph-phase notebooks).

In [ ]:
# ================================================================================
# SETUP: Install google-adk
# ================================================================================
# google-adk is Google's open-source Agent Development Kit. It bundles:
#   - google.adk.agents   -> Agent / LlmAgent and friends
#   - google.adk.tools    -> tool wrapping utilities
#   - google.adk.runners  -> local execution harness (Runner)
#   - google.adk.sessions -> conversation state management (SessionService)
# It also pulls in google-genai, which supplies the `types.Content` / `types.Part`
# message objects ADK uses under the hood.
# ================================================================================

!pip install -q google-adk

In [ ]:
# ================================================================================
# SETUP: Imports and environment
# ================================================================================
# GOOGLE_API_KEY is already a documented env var for this repo (see CLAUDE.md,
# where it is used for LangExtract Streamlit apps). The same key works here to
# authenticate ADK's default Gemini model access via the plain Gemini API
# (as opposed to going through Vertex AI, which uses GCP project/service-account
# credentials instead of an API key).
# ================================================================================

import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("GOOGLE_API_KEY"), (
    "Set GOOGLE_API_KEY in your .env file (a plain Gemini API key from Google AI "
    "Studio works — no Vertex AI project required for this notebook)."
)

In [ ]:
# ================================================================================
# STEP 1: Define a tool as a plain Python function
# ================================================================================
# ADK automatically wraps a plain Python function into a callable tool:
#   - the function name becomes the tool name
#   - the docstring becomes the tool description the model sees
#   - the type-hinted signature becomes the tool's input/output schema
# No manual JSON-schema authoring is required, similar in spirit to how
# LangChain/CrewAI auto-wrap functions decorated as tools.
# ================================================================================


def get_agentspace_fact(topic: str) -> dict:
    """Looks up a short factual note about Google AgentSpace or Google ADK.

    Args:
        topic: Either "agentspace" or "adk".

    Returns:
        A dict with a "status" key and a "fact" key containing the note.
    """
    facts = {
        "agentspace": (
            "AgentSpace is Google's enterprise platform for hosting, securing, "
            "and discovering agents, built on top of Vertex AI Agent Builder."
        ),
        "adk": (
            "ADK (Agent Development Kit) is Google's open-source framework for "
            "building, evaluating, and deploying AI agents in Python or Java."
        ),
    }
    key = topic.strip().lower()
    if key not in facts:
        return {"status": "error", "fact": f"No fact on file for topic '{topic}'."}
    return {"status": "success", "fact": facts[key]}

In [ ]:
# ================================================================================
# STEP 2: Build a minimal ADK Agent
# ================================================================================
# `Agent` is ADK's public, ergonomic alias for `LlmAgent` (google.adk.agents.Agent
# is google.adk.agents.LlmAgent). Required/relevant fields:
#   - name:        a unique identifier for this agent
#   - model:       a Gemini model id string (resolved through the Gemini API when
#                  GOOGLE_API_KEY is set, or through Vertex AI when configured for it)
#   - instruction: the system prompt describing the agent's role and behavior
#   - description: a short summary other agents/tools can use to route to this one
#   - tools:       a list of plain functions and/or ADK tool objects
# ================================================================================

from google.adk.agents import Agent

root_agent = Agent(
    name="agentspace_explainer_agent",
    model="gemini-2.0-flash",
    description="Answers questions about Google ADK and Google AgentSpace.",
    instruction=(
        "You are a concise technical explainer. When asked about Google ADK or "
        "Google AgentSpace, call the get_agentspace_fact tool to retrieve a "
        "grounded fact before answering, then explain it in one or two plain "
        "English sentences."
    ),
    tools=[get_agentspace_fact],
)

print(f"Created ADK agent: {root_agent.name} (model={root_agent.model})")

### Discussion of the Output

Notice that constructing an `Agent` does **not** call any API — it just builds a configuration object. Nothing runs until we hand this agent to a `Runner` with a session, which is the next step. This is the same separation of concerns you've seen in CrewAI (`Agent` vs. `Crew.kickoff()`) and LangGraph (building a graph vs. invoking it).

## 3. Running the Agent Locally with a Runner and Session

### What we are going to do

An ADK agent by itself is inert. To actually converse with it, ADK provides:
- a **`SessionService`** (here, `InMemorySessionService`) that creates and tracks a `Session` — the conversation's state and history, scoped by `app_name` / `user_id` / `session_id`;
- a **`Runner`** that binds an agent to a session service and drives the agent loop: it takes a `types.Content` user message, runs the agent (including any tool calls it decides to make), and yields a stream of `Event` objects as it goes.

We'll ask a question that should trigger the `get_agentspace_fact` tool, then walk the event stream for the final text response.

In [ ]:
# ================================================================================
# STEP 3: Create a session service and a session
# ================================================================================
# InMemorySessionService keeps session state in process memory -- fine for local
# development and this notebook. Production deployments (including on AgentSpace/
# Vertex AI) swap this for a persistent, managed session service instead.
# create_session is a coroutine, so we await it directly (Jupyter supports
# top-level await).
# ================================================================================

from google.adk.sessions import InMemorySessionService

APP_NAME = "agentspace_foundations_demo"
USER_ID = "demo_user"
SESSION_ID = "demo_session_01"

session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

print(f"Created session '{session.id}' for app '{APP_NAME}'")

In [ ]:
# ================================================================================
# STEP 4: Wire the agent to a Runner and send a message
# ================================================================================
# Runner.run(...) is a synchronous generator that yields Event objects as the
# agent thinks, calls tools, and responds. The final text answer is on the event
# for which event.is_final_response() is True.
# ================================================================================

from google.adk.runners import Runner
from google.genai import types

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

user_message = types.Content(
    role="user",
    parts=[types.Part(text="What is Google AgentSpace, in one sentence?")],
)

final_response_text = None

for event in runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=user_message):
    # Tool-call and tool-result events show up here too; we print author + a
    # short marker for each event, and capture the final natural-language answer.
    print(f"[event] author={event.author} final={event.is_final_response()}")
    if event.is_final_response() and event.content and event.content.parts:
        final_response_text = event.content.parts[0].text

print("\n--- Final agent response ---")
print(final_response_text)

### Discussion of the Output

Walking the event stream (rather than getting a single blocking return value) is a deliberate ADK design choice: it's the same mechanism that powers streaming responses, human-in-the-loop approval of tool calls, and live/voice agents — you get a uniform `Event` protocol whether the agent is answering in one shot or working through a multi-step tool-using turn. In our run, you should see at least one event where `event.author` is the agent making a tool call, followed by a final event carrying the natural-language answer that incorporates the fact returned by `get_agentspace_fact`.

## 4. From a Local Agent to AgentSpace (Conceptual)

### What we are going to do

This section is explanatory rather than runnable: deploying to AgentSpace requires a provisioned Google Cloud project, Vertex AI Agent Builder access, IAM/OAuth configuration for enterprise users, and (typically) a Vertex AI Agent Engine or Cloud Run endpoint to host the agent — all outside the scope of a local notebook. What follows is the conceptual path from the `root_agent` object we just built to something discoverable inside an organization's AgentSpace.

### The deployment path, at a glance

1. **Author the agent with ADK** (what we just did). The same `Agent`/`LlmAgent` object — with its `tools`, `instruction`, and optionally sub-agents — is the unit that gets deployed; nothing about the agent's code changes for deployment.
2. **Package and deploy to a hosting target.** ADK agents are typically deployed to one of:
   - **Vertex AI Agent Engine** — a managed, serverless runtime purpose-built for ADK (and other) agents, handling scaling, session state, and observability for you; or
   - **Cloud Run** — a containerized deployment (ADK includes CLI/config helpers, e.g. `adk deploy`) for teams that want more control over the runtime.
   Either way, the output of this step is a live, callable **agent endpoint** with its own identity in the GCP project.
3. **Register the deployed agent with AgentSpace / Vertex AI Agent Builder.** This is the step that turns "an endpoint only its developer knows about" into "an agent your organization can find." Registration attaches:
   - a **display name, description, and icon** end users see in the AgentSpace UI;
   - **IAM-backed access control**, so only the right users/groups can discover or invoke it;
   - **grounding/data-store bindings**, if the agent should search enterprise document stores AgentSpace already indexes.
4. **Discovery and use.** Once registered, the agent shows up alongside other agents and enterprise search results inside the AgentSpace experience — employees can converse with it, and administrators can monitor usage, audit conversations, and manage permissions centrally, without needing to know it was built with ADK specifically (AgentSpace can host non-ADK agents too, as long as they conform to its integration contract).

### Why this two-layer design matters

Separating "build framework" (ADK) from "hosting/discovery platform" (AgentSpace) mirrors a pattern you've already seen elsewhere in this repo: LangGraph agents can be run locally, deployed via LangGraph Platform, or wrapped in a FastAPI service (Phase 13's capstones) — the agent's *logic* doesn't change based on where it ends up running. The same ADK `Agent` you tested locally in Section 3 is, in principle, the same object a platform team would point `adk deploy` at to put in front of an entire enterprise via AgentSpace.

## 📖 Key Takeaways

- **Google ADK** is the open-source framework for *building* agents in Python/Java; **Google AgentSpace** is the enterprise platform, built on Vertex AI Agent Builder, for *hosting and discovering* agents — ADK is the code layer, AgentSpace is the distribution layer.
- A minimal ADK agent is an `Agent` (alias for `LlmAgent`) configured with `name`, `model`, `instruction`, and optionally `tools` — plain Python functions with type hints and docstrings are auto-wrapped into callable tools, no manual schema required.
- Running an agent locally requires a `SessionService` (e.g. `InMemorySessionService`) to hold conversation state and a `Runner` to drive the agent loop; `Runner.run(...)` yields a stream of `Event` objects, and the final natural-language answer is the one where `event.is_final_response()` is `True`.
- `GOOGLE_API_KEY` (already used elsewhere in this repo for LangExtract Streamlit apps) is sufficient to authenticate ADK's default Gemini model access outside of Vertex AI — no GCP project is required just to build and run an ADK agent locally.
- Moving an ADK agent to AgentSpace is a **deployment + registration** step, not a rewrite: the same agent object gets deployed to a hosting target (Vertex AI Agent Engine or Cloud Run) and then registered with AgentSpace/Agent Builder for IAM-governed enterprise discovery — this notebook covered that path conceptually, since it requires GCP enterprise setup outside this notebook's scope.

### 🎓 Next Steps

- Explore `google.adk.tools` for built-in tools (e.g. Google Search grounding) beyond plain-function tools.
- Try a multi-agent ADK setup with `sub_agents=[...]` and compare it to the LangGraph supervisor pattern in `07_Advanced_Agentic_Systems/Multi_Agent_Orchestration/`.
- Read the official [ADK documentation](https://google.github.io/adk-docs/) and the [Vertex AI Agent Builder / AgentSpace documentation](https://cloud.google.com/agentspace) for the full deployment and registration workflow.

### 📚 Additional Resources

- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [Google ADK GitHub Repository](https://github.com/google/adk-python)
- [Google Cloud AgentSpace Overview](https://cloud.google.com/agentspace)